In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import folium
from folium.plugins import HeatMap

In [3]:
# Load and clean data

df = pd.read_csv("sf_crime_2003_2025.csv")

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

df["lat"] = pd.to_numeric(df["lat"].astype(str).str.replace(",", "."), errors="coerce")
df["lon"] = pd.to_numeric(df["lon"].astype(str).str.replace(",", "."), errors="coerce")

df["district"] = df["district"].astype(str).str.strip().str.title()
df["crime"] = df["crime"].astype(str).str.strip()

df = df.dropna(subset=["datetime", "lat", "lon", "crime"])
df["year"] = df["datetime"].dt.year

df = df[df["year"] <= 2025]
df = df[df["lat"].between(37.6, 37.9) & df["lon"].between(-122.55, -122.35)]

os.makedirs("docs/images", exist_ok=True)
os.makedirs("docs/visualizations", exist_ok=True)



In [5]:
# 1. Static chart

yearly = df.groupby("year").size().reset_index(name="count")

plt.figure(figsize=(10, 5))
plt.plot(yearly["year"], yearly["count"], linewidth=2)
plt.title("Reported Crime Incidents in San Francisco by Year")
plt.xlabel("Year")
plt.ylabel("Incidents")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("docs/images/static_chart.png", dpi=300)
plt.close()



In [6]:
# 2. Map

sample = df.sample(min(len(df), 40000), random_state=42)

m = folium.Map(location=[37.7749, -122.4194], zoom_start=12, tiles="CartoDB positron")
HeatMap(sample[["lat", "lon"]].values.tolist(), radius=8, blur=10).add_to(m)
m.save("docs/visualizations/map.html")



In [7]:
# 3. Interactive Plotly

top_crimes = df["crime"].value_counts().head(5).index
tmp = df[df["crime"].isin(top_crimes)]

plot_df = tmp.groupby(["year", "crime"]).size().reset_index(name="count")

fig = px.line(
    plot_df,
    x="year",
    y="count",
    color="crime",
    markers=True,
    title="Top Crime Categories Over Time"
)

fig.update_layout(template="plotly_white", hovermode="x unified")
fig.write_html("docs/visualizations/interactive_chart.html")